In [1]:
import os
import json
import numpy as np
import pandas as pd
from astropy.time import Time
from IPython import get_ipython


def _load_search_runtime(search_dir='/home/msp25gd/ResearchProjectMSc/HR/search'):
    """Load ASSET helpers once in the current notebook kernel."""
    required = ['ASSET', 'to_reduce']
    missing = [name for name in required if name not in globals()]
    if len(missing) == 0:
        return

    ip = get_ipython()
    if ip is None:
        raise RuntimeError('This helper must run inside an IPython/Jupyter kernel.')

    for nb in ['asset.ipynb', 'reduce_name.ipynb']:
        nb_path = os.path.join(search_dir, nb)
        if not os.path.exists(nb_path):
            raise FileNotFoundError(f'Missing required notebook: {nb_path}')
        ip.run_line_magic('run', nb_path)


def _row_to_datetime(row):
    """Convert available row time fields into a regular UTC datetime."""
    # Priority 1: MJD
    if 'MJD-OBS' in row.index and pd.notna(row['MJD-OBS']):
        try:
            mjd_val = float(row['MJD-OBS'])
            return pd.Timestamp(Time(mjd_val, format='mjd', scale='utc').to_datetime())
        except Exception:
            pass

    # Priority 2: date-like columns
    for col in ['Date', 'DATE-OBS', 'DATE_OBS', 'Datetime', 'DATE']:
        if col in row.index and pd.notna(row[col]):
            dt = pd.to_datetime(row[col], errors='coerce', utc=True)
            if pd.notna(dt):
                return pd.Timestamp(dt)

    return pd.NaT


def get_star_detection_info(
    star_name,
    line='K',
    dataset_root='/home/msp25gd/ResearchProjectMSc/ResolutionHandling/processed_candidates/',
    param_path='/home/msp25gd/ResearchProjectMSc/HR/search/param.json',
    search_dir='/home/msp25gd/ResearchProjectMSc/HR/search',
):
    """
    Return three outputs for one star:
    1) summary dict with total spectra and number flagged with detection
    2) table for all spectra, including converted RegularDateTime
    3) table for detected spectra only
    """
    _load_search_runtime(search_dir=search_dir)

    if not os.path.exists(param_path):
        raise FileNotFoundError(f'param.json not found: {param_path}')
    if not os.path.isdir(dataset_root):
        raise FileNotFoundError(f'dataset_root not found: {dataset_root}')

    with open(param_path) as f:
        param = json.load(f)
    param['dataset'] = dataset_root if dataset_root.endswith('/') else dataset_root + '/'

    reduced = to_reduce(str(star_name)) if 'to_reduce' in globals() else str(star_name)
    star_path = os.path.join(param['dataset'], reduced)

    if not os.path.isdir(star_path):
        raise FileNotFoundError(
            f"Star folder not found in dataset: {star_path}. "
            f"Check reduced name '{reduced}' and dataset_root."
        )

    search = ASSET(parameters=param, line=line)
    search.ccf = False
    spec_param = search.spec_analysis(star_path + '/')

    if spec_param is None:
        raise ValueError(f'Not enough spectra to analyze detections for {reduced}.')

    new_spectra, med, med_err = spec_param

    if not hasattr(search, 'df') or search.df is None:
        raise RuntimeError('ASSET did not expose metadata table (search.df).')
    spectra_table = search.df.copy().reset_index(drop=True)

    if len(spectra_table) != len(new_spectra):
        raise RuntimeError(
            f'Row mismatch: metadata rows={len(spectra_table)} vs spectra={len(new_spectra)}'
        )

    detection_flags = []
    min_sigma_list = []
    rv_list = []
    width_list = []

    for idx, spec in enumerate(new_spectra):
        snr = search.snr(spec, med, search.spectra_err[idx], med_err)
        sd = np.std(snr)

        corr_snr = snr.copy()[search.snr_idxrange]
        sig = corr_snr / sd
        min_detect = float(np.nanmin(sig))

        filtered_rv = search.radial_velocity[search.snr_idxrange]
        rv_detect = float(filtered_rv[np.nanargmin(sig)])

        width = float(search.get_width(sig)) if min_detect < search.threshold else 0.0
        is_detection = (min_detect < search.threshold) and (width >= search.width_filter)

        detection_flags.append(bool(is_detection))
        min_sigma_list.append(min_detect)
        rv_list.append(rv_detect)
        width_list.append(width)

    spectra_table['SpectrumIndex'] = np.arange(1, len(spectra_table) + 1)
    spectra_table['Detection'] = detection_flags
    spectra_table['MinSigma'] = min_sigma_list
    spectra_table['RV_at_MinSigma'] = rv_list
    spectra_table['Width'] = width_list

    spectra_table['RegularDateTime'] = spectra_table.apply(_row_to_datetime, axis=1)
    spectra_table['DetectionDateTime'] = spectra_table['RegularDateTime'].where(spectra_table['Detection'])

    detection_table = spectra_table[spectra_table['Detection']].copy().reset_index(drop=True)

    summary = {
        'input_star': str(star_name),
        'reduced_star': str(reduced),
        'line': str(line),
        'total_spectra': int(len(spectra_table)),
        'detected_spectra': int(spectra_table['Detection'].sum()),
    }

    return summary, spectra_table, detection_table


# Example usage:
# summary, all_spectra_table, detected_only_table = get_star_detection_info('hd22049')
# print(summary)
# display(all_spectra_table)
# display(detected_only_table)